# Train SoundWave genre classifier

Notebook này train lại classifier head cho 12 genre của app. Nó giữ `discogs-effnet-bs64-1.pb` làm embedding extractor và export model mới dạng `.keras` để AI service load bằng `GENRE_CLASSIFIER_BACKEND=keras`.

Dataset cần đặt trong Google Drive theo dạng `dataset/<GenreName>/*.mp3`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/soundwave-genre-training')
DATASET_DIR = DRIVE_ROOT / 'dataset'
BASE_MODEL_DIR = DRIVE_ROOT / 'base_models'
CACHE_DIR = DRIVE_ROOT / 'cache'
OUTPUT_DIR = DRIVE_ROOT / 'output'

LABELS = ['Pop', 'Rap/Hip-Hop', 'R&B', 'Rock', 'Indie', 'EDM', 'Lo-Fi', 'Jazz', 'Acoustic', 'Bolero', 'Other']
DATASET_FOLDERS = {'Rap/Hip-Hop': 'Rap-Hip-Hop'}
AUDIO_EXTENSIONS = {'.mp3', '.wav', '.flac', '.m4a', '.ogg'}

EMBEDDING_MODEL_PATH = BASE_MODEL_DIR / 'discogs-effnet-bs64-1.pb'
EMBEDDING_OUTPUT_LAYER = 'PartitionedCall:1'
SAMPLE_RATE = 16000
MAX_SEGMENTS_PER_TRACK = 24
CACHE_FILE = CACHE_DIR / 'soundwave_discogs_effnet_segments_taxonomy_v2.npz'

print('Drive root:', DRIVE_ROOT)
print('Dataset dir:', DATASET_DIR)
print('Embedding model:', EMBEDDING_MODEL_PATH)


In [ ]:
!nvidia-smi || true
!pip -q install numpy pandas scikit-learn matplotlib librosa soundfile
!pip -q install essentia-tensorflow

import json
import os
import subprocess
import sys
import zipfile
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

print('Setup ready. Essentia embedding extraction will run in a separate Python process.')


In [ ]:
if not EMBEDDING_MODEL_PATH.exists():
    raise FileNotFoundError(f'Missing embedding model: {EMBEDDING_MODEL_PATH}')

rows = []
for label in LABELS:
    label_dir = DATASET_DIR / DATASET_FOLDERS.get(label, label)
    if not label_dir.exists():
        print(f'Warning: missing folder {label_dir}')
        continue
    for path in sorted(label_dir.rglob('*')):
        if path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS:
            rows.append({'path': str(path), 'label': label})

df = pd.DataFrame(rows)
if df.empty:
    raise RuntimeError('Không tìm thấy audio nào. Kiểm tra lại DATASET_DIR và folder genre.')

display(df.groupby('label').size().reindex(LABELS, fill_value=0).rename('tracks'))
print('Total tracks:', len(df))


In [ ]:
from pathlib import Path
from collections import Counter
import subprocess
import sys
import numpy as np

MYDRIVE_DIR = Path('/content/drive/MyDrive')
if not MYDRIVE_DIR.is_dir():
    raise RuntimeError('Google Drive is not mounted. Run the Drive mount cell above first.')

if 'DRIVE_ROOT' not in globals():
    DRIVE_ROOT = MYDRIVE_DIR / 'soundwave-genre-training'
if 'DATASET_DIR' not in globals():
    DATASET_DIR = DRIVE_ROOT / 'dataset'
if 'BASE_MODEL_DIR' not in globals():
    BASE_MODEL_DIR = DRIVE_ROOT / 'base_models'
if 'CACHE_DIR' not in globals():
    CACHE_DIR = DRIVE_ROOT / 'cache'
if 'LABELS' not in globals():
    LABELS = ['Pop', 'Rap/Hip-Hop', 'R&B', 'Rock', 'Indie', 'EDM', 'Lo-Fi', 'Jazz', 'Acoustic', 'Bolero', 'Other']
if 'DATASET_FOLDERS' not in globals():
    DATASET_FOLDERS = {'Rap/Hip-Hop': 'Rap-Hip-Hop'}
if 'EMBEDDING_MODEL_PATH' not in globals():
    EMBEDDING_MODEL_PATH = BASE_MODEL_DIR / 'discogs-effnet-bs64-1.pb'
if 'EMBEDDING_OUTPUT_LAYER' not in globals():
    EMBEDDING_OUTPUT_LAYER = 'PartitionedCall:1'
if 'SAMPLE_RATE' not in globals():
    SAMPLE_RATE = 16000
if 'MAX_SEGMENTS_PER_TRACK' not in globals():
    MAX_SEGMENTS_PER_TRACK = 24
if 'CACHE_FILE' not in globals():
    CACHE_FILE = CACHE_DIR / 'soundwave_discogs_effnet_segments_taxonomy_v2.npz'

if not DRIVE_ROOT.is_dir():
    raise FileNotFoundError(f'Drive root does not exist: {DRIVE_ROOT}')
if not DATASET_DIR.is_dir():
    raise FileNotFoundError(f'Dataset folder does not exist: {DATASET_DIR}')
if not EMBEDDING_MODEL_PATH.is_file():
    raise FileNotFoundError(f'Embedding model file does not exist: {EMBEDDING_MODEL_PATH}')

CACHE_DIR.mkdir(exist_ok=True)
REBUILD_EMBEDDING_CACHE = False
if REBUILD_EMBEDDING_CACHE and CACHE_FILE.exists():
    CACHE_FILE.unlink()

EXTRACTOR_SCRIPT = Path('/content/soundwave_extract_embeddings.py')
EXTRACTOR_SCRIPT.write_text(r'''
import argparse
import json
from pathlib import Path

import numpy as np

LABELS = ['Pop', 'Rap/Hip-Hop', 'R&B', 'Rock', 'Indie', 'EDM', 'Lo-Fi', 'Jazz', 'Acoustic', 'Bolero', 'Other']
DATASET_FOLDERS = {'Rap/Hip-Hop': 'Rap-Hip-Hop'}
AUDIO_EXTENSIONS = {'.mp3', '.wav', '.flac', '.m4a', '.ogg'}

def sample_segments(embeddings, max_segments):
    embeddings = np.asarray(embeddings, dtype=np.float32)
    if embeddings.ndim == 1:
        embeddings = embeddings.reshape(1, -1)
    if len(embeddings) <= max_segments:
        return embeddings
    indices = np.linspace(0, len(embeddings) - 1, max_segments).round().astype(int)
    return embeddings[indices]

def scan_audio(dataset_dir):
    rows = []
    for label in LABELS:
        label_dir = dataset_dir / DATASET_FOLDERS.get(label, label)
        if not label_dir.exists():
            print('Warning: missing folder', label_dir, flush=True)
            continue
        for path in sorted(label_dir.rglob('*')):
            if path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS:
                rows.append((path, label))
    return rows

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--dataset-dir', required=True)
    parser.add_argument('--model-path', required=True)
    parser.add_argument('--output-layer', required=True)
    parser.add_argument('--cache-file', required=True)
    parser.add_argument('--failures-file', required=True)
    parser.add_argument('--sample-rate', type=int, required=True)
    parser.add_argument('--max-segments', type=int, required=True)
    args = parser.parse_args()

    print('Scanning dataset:', args.dataset_dir, flush=True)
    rows = scan_audio(Path(args.dataset_dir))
    if not rows:
        raise RuntimeError('No audio files found for embedding extraction.')

    print(f'Found {len(rows)} audio file(s).', flush=True)
    print('Importing Essentia TensorFlow algorithms...', flush=True)
    from essentia.standard import MonoLoader, TensorflowPredictEffnetDiscogs
    print('Loading embedding model:', args.model_path, flush=True)
    embedding_model = TensorflowPredictEffnetDiscogs(
        graphFilename=args.model_path,
        output=args.output_layer,
    )
    print('Embedding model loaded.', flush=True)
    X, y, track_ids, file_paths, failures = [], [], [], [], []

    for idx, (audio_path, label) in enumerate(rows):
        track_id = f'{label}/{audio_path.stem}/{idx}'
        try:
            print(f'[{idx + 1}/{len(rows)}] {label}: {audio_path.name}', flush=True)
            audio = MonoLoader(filename=str(audio_path), sampleRate=args.sample_rate, resampleQuality=4)()
            segments = sample_segments(embedding_model(audio), args.max_segments)
            for segment in segments:
                X.append(segment.astype(np.float32))
                y.append(label)
                track_ids.append(track_id)
                file_paths.append(str(audio_path))
            print(f'  -> {len(segments)} segment(s), cached segments: {len(X)}', flush=True)
        except Exception as exc:
            failures.append((str(audio_path), str(exc)))
            print('Failed:', audio_path, exc, flush=True)

    if not X:
        raise RuntimeError('Embedding extraction produced no segments.')

    cache_file = Path(args.cache_file)
    cache_file.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        cache_file,
        X=np.asarray(X, dtype=np.float32),
        y=np.asarray(y),
        track_ids=np.asarray(track_ids),
        file_paths=np.asarray(file_paths),
    )
    Path(args.failures_file).write_text(json.dumps(failures, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Saved embedding cache:', cache_file, flush=True)

if __name__ == '__main__':
    main()
'''.strip() + '\n', encoding='utf-8')

if not CACHE_FILE.exists():
    extractor_command = [
        sys.executable,
        '-u',
        str(EXTRACTOR_SCRIPT),
        '--dataset-dir', str(DATASET_DIR),
        '--model-path', str(EMBEDDING_MODEL_PATH),
        '--output-layer', EMBEDDING_OUTPUT_LAYER,
        '--cache-file', str(CACHE_FILE),
        '--failures-file', str(CACHE_DIR / 'embedding_failures.json'),
        '--sample-rate', str(SAMPLE_RATE),
        '--max-segments', str(MAX_SEGMENTS_PER_TRACK),
    ]
    print('Running embedding extractor...', flush=True)
    extractor_process = subprocess.Popen(
        extractor_command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    ignored_extractor_log_patterns = (
        'No network created, or last created network has been deleted',
    )
    if extractor_process.stdout is None:
        raise RuntimeError('Embedding extractor stdout was not available.')
    for line in extractor_process.stdout:
        if any(pattern in line for pattern in ignored_extractor_log_patterns):
            continue
        print(line, end='', flush=True)
    extractor_exit_code = extractor_process.wait()
    if extractor_exit_code != 0:
        raise RuntimeError(f'Embedding extractor failed with exit code {extractor_exit_code}.')
else:
    print('Loading cached embeddings:', CACHE_FILE)

cache = np.load(CACHE_FILE, allow_pickle=True)
X, y, track_ids, file_paths = cache['X'], cache['y'], cache['track_ids'], cache['file_paths']
print('Segments:', X.shape)
print('Segment labels:', Counter(y))


In [ ]:
unique_tracks = np.unique(track_ids)
track_to_label = {tid: y[np.where(track_ids == tid)[0][0]] for tid in unique_tracks}
track_labels = np.asarray([track_to_label[tid] for tid in unique_tracks])

counts = Counter(track_labels)
stratify_tracks = track_labels if min(counts.values()) >= 2 else None
train_ids, temp_ids = train_test_split(
    unique_tracks,
    test_size=0.30,
    random_state=42,
    stratify=stratify_tracks,
)

temp_labels = np.asarray([track_to_label[tid] for tid in temp_ids])
temp_counts = Counter(temp_labels)
stratify_temp = temp_labels if min(temp_counts.values()) >= 2 else None
val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=42,
    stratify=stratify_temp,
)

def mask_for(ids):
    return np.isin(track_ids, ids)

label_to_idx = {label: idx for idx, label in enumerate(LABELS)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}

def encode(labels):
    return np.asarray([label_to_idx[label] for label in labels], dtype=np.int64)

train_mask, val_mask, test_mask = mask_for(train_ids), mask_for(val_ids), mask_for(test_ids)
X_train, y_train = X[train_mask], encode(y[train_mask])
X_val, y_val = X[val_mask], encode(y[val_mask])
X_test, y_test = X[test_mask], encode(y[test_mask])

print('Train segments:', X_train.shape, Counter(y[train_mask]))
print('Val segments:', X_val.shape, Counter(y[val_mask]))
print('Test segments:', X_test.shape, Counter(y[test_mask]))


In [ ]:
import tensorflow as tf

print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
tf.keras.utils.set_random_seed(42)

num_classes = len(LABELS)
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1280,), name='discogs_effnet_embedding'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.35),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Dense(num_classes, activation='softmax', name='genre_probs'),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

present_classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=present_classes, y=y_train)
class_weight = {int(cls): float(weight) for cls, weight in zip(present_classes, weights)}
print('Class weights:', {idx_to_label[k]: round(v, 3) for k, v in class_weight.items()})

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=128,
    class_weight=class_weight,
    callbacks=callbacks,
)


In [ ]:
test_probs = model.predict(X_test, verbose=0)
test_pred = test_probs.argmax(axis=1)
print(classification_report(y_test, test_pred, target_names=LABELS, zero_division=0))
print('Confusion matrix:')
print(confusion_matrix(y_test, test_pred))

def evaluate_per_track(ids):
    y_true, y_pred = [], []
    for tid in ids:
        mask = track_ids == tid
        probs = model.predict(X[mask], verbose=0).mean(axis=0)
        y_true.append(label_to_idx[track_to_label[tid]])
        y_pred.append(int(probs.argmax()))
    print(classification_report(y_true, y_pred, target_names=LABELS, zero_division=0))

print('Per-track test report:')
evaluate_per_track(test_ids)


In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True)

MODEL_NAME = 'soundwave_genre_classifier'
KERAS_PATH = OUTPUT_DIR / f'{MODEL_NAME}.keras'
METADATA_PATH = OUTPUT_DIR / f'{MODEL_NAME}.json'
ZIP_PATH = OUTPUT_DIR / f'{MODEL_NAME}.zip'

model.save(KERAS_PATH)

metadata = {
    'name': 'soundwave_genre_classifier-v1',
    'type': 'single-label classifier',
    'description': 'SoundWave app genre classifier trained on Discogs EffNet embeddings',
    'framework': 'tensorflow.keras',
    'embedding_model': 'discogs-effnet-bs64-1',
    'classifier_backend': 'keras',
    'input_shape': [1280],
    'classes': LABELS,
    'training': {
        'tracks': int(len(unique_tracks)),
        'segments': int(len(X)),
        'max_segments_per_track': int(MAX_SEGMENTS_PER_TRACK),
        'sample_rate': int(SAMPLE_RATE),
    },
}
with open(METADATA_PATH, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(KERAS_PATH, arcname=KERAS_PATH.name)
    zf.write(METADATA_PATH, arcname=METADATA_PATH.name)

print('Exported:')
print(KERAS_PATH)
print(METADATA_PATH)
print(ZIP_PATH)
